# Data audit

This notebook inspects data integrity, studies bounding box distributions and extreme cases, and searches for exact or near-duplicate images before any train/validation split. Heavy reusable logic lives in `src/turtle_detection`; this notebook only orchestrates the analysis and displays results.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from turtle_detection.box_analysis import (
    add_box_features,
    invalid_box_mask,
    select_extreme_boxes,
)
from turtle_detection.image_quality import analyze_image_quality
from turtle_detection.image_similarity import (
    build_similarity_groups,
    compute_perceptual_hashes,
    find_similar_pairs,
    select_reviewed_pairs,
)

RAW_DIR = PROJECT_ROOT / "data/raw"
IMAGE_DIR = RAW_DIR / "IMAGES_512"
SEED = 42
from turtle_detection.splitting import (
    assign_split_groups,
    assert_no_group_leakage,
    create_group_shuffle_split,
    save_split,
    select_distribution_extremes,
    summarize_categorical_split_balance,
    summarize_continuous_split_balance,
)


## 1. CSV inspection

In [ ]:
train_df = pd.read_csv(RAW_DIR / "Train.csv")
submission_df = pd.read_csv(RAW_DIR / "SampleSubmission.csv")

for name, dataframe in {"Train": train_df, "SampleSubmission": submission_df}.items():
    print(f"\n{name}: {dataframe.shape}")
    display(dataframe.head())
    display(dataframe.dtypes.rename("dtype").to_frame())
    display(dataframe.isna().sum().rename("missing").to_frame())
    print("Duplicate rows:", dataframe.duplicated().sum())
    print("Duplicate Image_ID values:", dataframe["Image_ID"].duplicated().sum())

## 2. Image inventory and integrity

In [ ]:
image_paths = sorted(path for path in IMAGE_DIR.iterdir() if path.is_file())
image_sizes = Counter()
image_modes = Counter()
unreadable = []
sha256_hashes = defaultdict(list)

for image_path in image_paths:
    try:
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            image_sizes[image.size] += 1
            image_modes[image.mode] += 1
    except Exception as error:
        unreadable.append((image_path.name, str(error)))
    sha256_hashes[hashlib.sha256(image_path.read_bytes()).hexdigest()].append(image_path.name)

exact_duplicate_groups = [names for names in sha256_hashes.values() if len(names) > 1]

print("Images:", len(image_paths))
print("Extensions:", Counter(path.suffix for path in image_paths))
print("Resolutions:", image_sizes)
print("Modes:", image_modes)
print("Unreadable files:", unreadable)
print("Exact duplicates:", exact_duplicate_groups)

## 3. Table-to-image correspondence

In [ ]:
image_ids = {path.stem for path in image_paths}
train_ids = set(train_df["Image_ID"].astype(str))
test_ids = set(submission_df["Image_ID"].astype(str))

print("Train/test overlap:", sorted(train_ids & test_ids))
print("Training rows without images:", sorted(train_ids - image_ids))
print("Test rows without images:", sorted(test_ids - image_ids))
print("Orphan images:", sorted(image_ids - train_ids - test_ids))

## 4. Bounding box validation and derived features

The derived features describe box size, shape, position, border proximity, and distance from the image center.

In [ ]:
box_features = add_box_features(train_df)
invalid_boxes = invalid_box_mask(train_df)

print("Invalid boxes:", invalid_boxes.sum())
display(box_features[[
    "x", "y", "w", "h",
    "center_x", "center_y",
    "relative_area", "box_aspect_ratio",
    "distance_to_border", "distance_to_center",
]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

## 5. Bounding box distributions

In [ ]:
distribution_columns = [
    "w", "h", "relative_area",
    "box_aspect_ratio", "distance_to_border", "distance_to_center",
]
figure, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, column in zip(axes.flat, distribution_columns, strict=True):
    axis.hist(box_features[column], bins=40, edgecolor="white")
    axis.set_title(column.replace("_", " "))
    axis.set_ylabel("count")
figure.tight_layout()
plt.show()

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(
    box_features["center_x"],
    box_features["center_y"],
    c=box_features["relative_area"],
    cmap="viridis",
    alpha=0.65,
    s=18,
)
axes[0].set(
    title="Box centers and relative area",
    xlabel="center_x",
    ylabel="center_y",
    xlim=(0, 1),
    ylim=(1, 0),
)

heatmap = axes[1].hist2d(
    box_features["center_x"],
    box_features["center_y"],
    bins=20,
    range=[[0, 1], [0, 1]],
    cmap="magma",
)
axes[1].set(
    title="Box center density",
    xlabel="center_x",
    ylabel="center_y",
    ylim=(1, 0),
)
figure.colorbar(heatmap[3], ax=axes[1], label="count")
figure.tight_layout()
plt.show()

## 6. Visual review helpers

In [ ]:
def show_box_gallery(dataframe, title, columns=5, max_images=20):
    rows = list(dataframe.head(max_images).itertuples(index=False))
    row_count = (len(rows) + columns - 1) // columns
    figure, axes = plt.subplots(row_count, columns, figsize=(4 * columns, 3.2 * row_count))
    axes = list(getattr(axes, "flat", [axes]))

    for axis, row in zip(axes, rows):
        with Image.open(IMAGE_DIR / f"{row.Image_ID}.JPG") as source:
            image = source.convert("RGB")
        width, height = image.size
        axis.imshow(image)
        axis.add_patch(Rectangle(
            (row.x * width, row.y * height),
            row.w * width,
            row.h * height,
            fill=False,
            edgecolor="lime",
            linewidth=2,
        ))
        axis.set_title(str(row.Image_ID))
        axis.axis("off")

    for axis in axes[len(rows):]:
        axis.axis("off")
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()

## 7. Random annotations

In [ ]:
random_sample = train_df.sample(n=20, random_state=SEED)
show_box_gallery(random_sample, "Random annotation sample")

## 8. Extreme bounding box cases

Review each category visually. A statistical extreme may be a legitimate hard example or an annotation problem.

In [ ]:
extreme_cases = select_extreme_boxes(train_df, count=20)
list(extreme_cases)

In [ ]:
# Change the key to review each extreme category.
extreme_category = "smallest"
show_box_gallery(
    extreme_cases[extreme_category],
    f"Extreme boxes: {extreme_category}",
)

## 9. Near-duplicate candidates with perceptual hashing

SHA-256 only detects identical files. Perceptual hashing can find recompressed or minimally transformed copies. The Hamming-distance threshold is a candidate generator, not ground truth. Review the closest pairs visually before assigning groups.

In [ ]:
perceptual_hashes = compute_perceptual_hashes(image_paths, hash_size=8)
similar_pairs = find_similar_pairs(perceptual_hashes, max_distance=6)

print("Candidate pairs:", len(similar_pairs))
display(similar_pairs.head(30))

In [ ]:
if not similar_pairs.empty:
    similar_pairs["phash_distance"].value_counts().sort_index().plot.bar(figsize=(10, 4))
    plt.title("Perceptual-hash distance among candidate pairs")
    plt.xlabel("Hamming distance")
    plt.ylabel("pair count")
    plt.show()

## 10. Visual review of near-duplicate pairs

In [ ]:
def show_pair_gallery(pairs, max_pairs=12):
    reviewed = pairs.head(max_pairs)
    if reviewed.empty:
        print("No perceptual-hash candidates found at this threshold.")
        return
    figure, axes = plt.subplots(len(reviewed), 2, figsize=(10, 4 * len(reviewed)))
    if len(reviewed) == 1:
        axes = [axes]

    for axis_pair, pair in zip(axes, reviewed.itertuples(index=False)):
        for axis, image_id in zip(axis_pair, [pair.left_id, pair.right_id], strict=True):
            with Image.open(IMAGE_DIR / f"{image_id}.JPG") as source:
                axis.imshow(source.convert("RGB"))
            axis.set_title(f"{image_id} | distance={pair.phash_distance}")
            axis.axis("off")
    figure.tight_layout()
    plt.show()

show_pair_gallery(similar_pairs, max_pairs=12)

## 11. Confirm related pairs and build groups

A pair is confirmed only when all of the following hold:

1. It is byte-identical, or has pHash distance at most 6.
2. Visual review confirms the same turtle, scene, pose, background, and identification card.
3. Differences are limited to compression, exposure, framing, or a consecutive capture.
4. The pair is not accepted from pHash alone.

The two exact orphan/train pairs are excluded from grouping because orphan images are not training rows. The eight train/train pairs below were visually reviewed and confirmed as related captures. Their image IDs are recorded here so the grouping decisions travel with the notebook; they identify downloaded images, not individual turtles.

In [ ]:
# Image IDs refer to downloaded images, not turtle identities.
# Keep manually reviewed related captures in one split group to prevent leakage.
REVIEWED_RELATED_PAIRS = {
    ('0C74D523', '62958DCC'),
    ('15682678', '6EDACDED'),
    ('5676AA24', '6D4F4A2B'),
    ('5CE52D04', 'C4F4B9B9'),
    ('7DAAD9D8', '8D87431C'),
    ('9134F66A', 'F5CDE434'),
    ('B49734C7', 'DEEB7B8F'),
    ('F4D66974', 'FB83C812'),
}

reviewed_pairs = select_reviewed_pairs(similar_pairs, REVIEWED_RELATED_PAIRS)
similarity_groups = build_similarity_groups(train_ids, reviewed_pairs)

print("Confirmed related pairs:", len(reviewed_pairs))
print("Images in multi-image groups:", (similarity_groups["similarity_group"] >= 0).sum())
display(reviewed_pairs)
display(similarity_groups.query("similarity_group >= 0"))

## 12. Assign a unique group to every independent image

Each confirmed component keeps one shared group ID. Every unrelated image receives its own group ID. This is essential: using `-1` for every singleton would incorrectly make all independent images one giant group.

In [ ]:
split_groups = assign_split_groups(similarity_groups)

print("Training images:", len(split_groups))
print("Unique split groups:", split_groups["group_id"].nunique())
print("Multi-image groups:", reviewed_pairs.shape[0])
print("Singleton groups:", (split_groups["similarity_group"] < 0).sum())
display(split_groups.head())

## 13. Run and validate GroupShuffleSplit

`GroupShuffleSplit` samples complete groups, not individual rows. The validation fraction is therefore approximate when groups contain different numbers of images.

In [ ]:
split_assignments = create_group_shuffle_split(
    train_df,
    split_groups,
    validation_size=0.20,
    random_state=42,
)
assert_no_group_leakage(split_assignments)

split_summary = (
    split_assignments.groupby("split")
    .agg(images=("Image_ID", "size"), groups=("group_id", "nunique"))
)
split_summary["image_fraction"] = split_summary["images"] / len(split_assignments)
display(split_summary)

train_group_ids = set(split_assignments.query("split == 'train'")["group_id"])
validation_group_ids = set(split_assignments.query("split == 'validation'")["group_id"])
print("Leaking groups:", sorted(train_group_ids & validation_group_ids))

## 14. Save the reproducible split

The saved CSV contains `Image_ID`, `group_id`, and `split`. Raw images remain untouched.

In [ ]:
SPLIT_PATH = PROJECT_ROOT / "data/splits/train_validation.csv"
save_split(split_assignments, SPLIT_PATH)
print(f"Saved split to: {SPLIT_PATH}")

## 15. Image geometry and basic visual quality

Compute geometry and three relative quality indicators for every labeled image:

- `brightness`: mean normalized luminance;
- `contrast`: robust 95th-to-5th luminance percentile range;
- `sharpness`: variance of the luminance Laplacian.

Sharpness depends on resolution and scene texture. Use it to rank images for visual review, not as an automatic blur label or exclusion rule.


In [ ]:
training_image_paths = [
    IMAGE_DIR / f"{image_id}.JPG" for image_id in train_df["Image_ID"]
]
image_quality = analyze_image_quality(training_image_paths)

split_audit = (
    box_features
    .merge(split_assignments, on="Image_ID", how="left", validate="one_to_one")
    .merge(image_quality, on="Image_ID", how="left", validate="one_to_one")
)
split_audit["image_resolution"] = (
    split_audit["image_width"].astype("Int64").astype(str)
    + "x"
    + split_audit["image_height"].astype("Int64").astype(str)
)

required_audit_columns = [
    "split",
    "image_width",
    "image_height",
    "image_aspect_ratio",
    "brightness",
    "contrast",
    "sharpness",
]
assert len(split_audit) == len(train_df)
assert not split_audit[required_audit_columns].isna().any().any()

display(
    split_audit.groupby("split")[["brightness", "contrast", "sharpness"]]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
)


## 16. Train and validation representativeness

Compare both partitions without requiring identical distributions. The standardized mean difference measures mean displacement in pooled standard-deviation units. The normalized Wasserstein distance measures the overall distribution displacement relative to the pooled interquartile range, falling back to pooled standard deviation when the interquartile range is zero.

Large values are review signals, not automatic failure thresholds. Any split change must preserve groups and must be selected from data characteristics only, never from model validation scores.


In [ ]:
geometry_balance_columns = [
    "relative_area",
    "box_aspect_ratio",
    "center_x",
    "center_y",
    "distance_to_border",
    "distance_to_center",
    "image_aspect_ratio",
]
quality_balance_columns = ["brightness", "contrast", "sharpness"]

geometry_balance = summarize_continuous_split_balance(
    split_audit,
    geometry_balance_columns,
)
quality_balance = summarize_continuous_split_balance(
    split_audit,
    quality_balance_columns,
)
resolution_balance = summarize_categorical_split_balance(
    split_audit,
    "image_resolution",
)

print("Geometry balance")
display(geometry_balance.round(4))
print("Image quality balance")
display(quality_balance.round(4))
print("Resolution proportions")
display(resolution_balance.round(4))

sharpness_by_resolution = (
    split_audit.groupby(["image_resolution", "split"])["sharpness"]
    .agg(["count", "mean", "median", "std"])
)
print("Sharpness within each resolution")
display(sharpness_by_resolution.round(6))


In [ ]:
def plot_split_ecdfs(dataframe, columns, columns_per_row=3):
    rows = (len(columns) + columns_per_row - 1) // columns_per_row
    figure, axes = plt.subplots(
        rows,
        columns_per_row,
        figsize=(5 * columns_per_row, 3.8 * rows),
        squeeze=False,
    )
    for axis, column in zip(axes.flat, columns):
        for split, color in [("train", "tab:blue"), ("validation", "tab:orange")]:
            values = dataframe.loc[dataframe["split"].eq(split), column].sort_values()
            cumulative_probability = pd.Series(range(1, len(values) + 1)) / len(values)
            axis.plot(values.to_numpy(), cumulative_probability, label=split, color=color)
        axis.set_title(column.replace("_", " ").title())
        axis.set_xlabel(column)
        axis.set_ylabel("Cumulative probability")
        axis.grid(alpha=0.2)
    for axis in axes.flat[len(columns):]:
        axis.set_visible(False)
    axes.flat[0].legend()
    figure.tight_layout()
    return figure

plot_split_ecdfs(split_audit, geometry_balance_columns)
plt.show()
plot_split_ecdfs(split_audit, quality_balance_columns)
plt.show()


## 17. Global extremes and split coverage

Define every extreme from the full labeled distribution, then count and inspect the same global tails in each partition. This prevents each split from defining its own convenient version of an extreme case. Ties at a percentile threshold may select more than exactly 2.5% of the images; this is expected for features such as zero distance to a border.


In [ ]:
extreme_criteria = {
    "small_box": ("relative_area", "low"),
    "large_box": ("relative_area", "high"),
    "vertical_box": ("box_aspect_ratio", "low"),
    "horizontal_box": ("box_aspect_ratio", "high"),
    "near_border": ("distance_to_border", "low"),
    "off_center": ("distance_to_center", "high"),
    "dark_image": ("brightness", "low"),
    "bright_image": ("brightness", "high"),
    "low_contrast": ("contrast", "low"),
    "low_sharpness": ("sharpness", "low"),
}

split_extremes, extreme_thresholds = select_distribution_extremes(
    split_audit,
    extreme_criteria,
    tail_fraction=0.025,
)
extreme_rows = pd.concat(
    [rows.assign(extreme_case=name) for name, rows in split_extremes.items()],
    ignore_index=True,
)
extreme_counts = (
    extreme_rows.groupby(["extreme_case", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["train", "validation"], fill_value=0)
)
split_sizes = split_audit["split"].value_counts()
extreme_proportions = extreme_counts.div(split_sizes, axis="columns")

print("Global thresholds")
display(extreme_thresholds.round(6))
print("Extreme counts by split")
display(extreme_counts)
print("Extreme proportions by split")
display(extreme_proportions.round(4))


In [ ]:
# Change this value to review each global extreme category.
EXTREME_CASE_TO_REVIEW = "low_sharpness"

for split in ["train", "validation"]:
    review_rows = split_extremes[EXTREME_CASE_TO_REVIEW].query("split == @split")
    show_box_gallery(
        review_rows,
        title=f"{EXTREME_CASE_TO_REVIEW.replace('_', ' ').title()} - {split}",
        max_images=12,
    )


## 18. Session-related images beyond pHash

pHash is effective for near-identical images but may miss consecutive photographs of the same turtle or scene. If visual review shows that this remains a leakage risk, the next step is to compute pretrained CNN embeddings, retrieve nearest neighbors by cosine similarity, and review those pairs. That step requires PyTorch and is intentionally deferred until pHash results justify the added dependency and complexity. Image metadata must not be used as a model feature or submission signal.

## Conclusions checklist

After running and visually reviewing all sections:

- [x] Bounding-box validity, size, shape, and position distributions inspected.
- [x] Geometric extreme cases inspected.
- [x] Related images grouped and split leakage ruled out.
- [x] Train and validation geometry distributions compared.
- [x] Resolution proportions compared between train and validation.
- [x] Brightness, contrast, and sharpness distributions compared.
- [x] Global geometric and quality extremes reviewed in both splits.
- [x] Current group-safe split accepted; no seed change is justified.
- [ ] Need for CNN embedding analysis decided.
